# Exploration of Sample Stream (NHDPlus) Dataset

In [1]:
from pathlib import Path
import zipfile
import pandas as pd

ROOT = Path("/courses/DS5500.202710/teams/team4")
NHD_DIR = ROOT / "data/raw/nhdplus"
EXTRACT_DIR = ROOT / "data/processed/nhdplus_extracted"

packages = sorted(NHD_DIR.glob("*.zip"))

print(f"Found {len(packages)} ZIP packages\n")

for p in packages:
    print(f"{p.name}: {p.stat().st_size / 1e6:.1f} MB")
    print(f"  Valid ZIP: {zipfile.is_zipfile(p)}")

Found 5 ZIP packages

NHDPLUS_H_0313_HU4_GDB.zip: 241.0 MB
  Valid ZIP: True
NHDPLUS_H_0315_HU4_GDB.zip: 334.5 MB
  Valid ZIP: True
NHDPLUS_H_0316_HU4_GDB.zip: 323.2 MB
  Valid ZIP: True
NHDPLUS_H_19020401_HU8_GDB.zip: 22.8 MB
  Valid ZIP: True
NHDPLUS_H_19020503_HU8_GDB.zip: 53.1 MB
  Valid ZIP: True


In [2]:
for package in packages:
    print(f"\n{'=' * 75}\n{package.name}")

    with zipfile.ZipFile(package) as z:
        members = z.namelist()
        print(f"Total files inside archive: {len(members):,}")

        # Show the first 25 paths
        for name in members[:25]:
            print(" ", name)

        # Identify File Geodatabases
        gdbs = sorted({
            "/".join(name.split("/")[:i + 1])
            for name in members
            for i, part in enumerate(name.split("/"))
            if part.lower().endswith(".gdb")
        })

        print("\nGeodatabase directories:")
        for gdb in gdbs:
            print(" ", gdb)


NHDPLUS_H_0313_HU4_GDB.zip
Total files inside archive: 372
  NHDPLUS_H_0313_HU4_GDB.gdb/a00000001.freelist
  NHDPLUS_H_0313_HU4_GDB.gdb/a00000001.gdbindexes
  NHDPLUS_H_0313_HU4_GDB.gdb/a00000001.gdbtable
  NHDPLUS_H_0313_HU4_GDB.gdb/a00000001.gdbtablx
  NHDPLUS_H_0313_HU4_GDB.gdb/a00000001.TablesByName.atx
  NHDPLUS_H_0313_HU4_GDB.gdb/a00000002.gdbtable
  NHDPLUS_H_0313_HU4_GDB.gdb/a00000002.gdbtablx
  NHDPLUS_H_0313_HU4_GDB.gdb/a00000003.gdbindexes
  NHDPLUS_H_0313_HU4_GDB.gdb/a00000003.gdbtable
  NHDPLUS_H_0313_HU4_GDB.gdb/a00000003.gdbtablx
  NHDPLUS_H_0313_HU4_GDB.gdb/a00000004.CatItemsByPhysicalName.atx
  NHDPLUS_H_0313_HU4_GDB.gdb/a00000004.CatItemsByType.atx
  NHDPLUS_H_0313_HU4_GDB.gdb/a00000004.FDO_UUID.atx
  NHDPLUS_H_0313_HU4_GDB.gdb/a00000004.freelist
  NHDPLUS_H_0313_HU4_GDB.gdb/a00000004.gdbindexes
  NHDPLUS_H_0313_HU4_GDB.gdb/a00000004.gdbtable
  NHDPLUS_H_0313_HU4_GDB.gdb/a00000004.gdbtablx
  NHDPLUS_H_0313_HU4_GDB.gdb/a00000004.spx
  NHDPLUS_H_0313_HU4_GDB.gdb/a00000

## Extract one zip file for inspection

In [3]:
pilot_zip = NHD_DIR / "NHDPLUS_H_19020401_HU8_GDB.zip"
extract_path = EXTRACT_DIR / pilot_zip.stem
extract_path.mkdir(parents=True, exist_ok=True)

if not any(extract_path.iterdir()):
    with zipfile.ZipFile(pilot_zip) as z:
        z.extractall(extract_path)

print(f"Extracted to: {extract_path}")

gdb_paths = sorted(
    p for p in extract_path.rglob("*")
    if p.is_dir() and p.suffix.lower() == ".gdb"
)

for path in gdb_paths:
    print(path)

Extracted to: /courses/DS5500.202710/teams/team4/data/processed/nhdplus_extracted/NHDPLUS_H_19020401_HU8_GDB
/courses/DS5500.202710/teams/team4/data/processed/nhdplus_extracted/NHDPLUS_H_19020401_HU8_GDB/NHDPLUS_H_19020401_HU8_GDB.gdb


## List all layers in GDB

In [4]:
import geopandas as gpd

if not gdb_paths:
    raise FileNotFoundError("No .gdb directory found in the extracted package.")

gdb_path = gdb_paths[0]
layers = gpd.list_layers(gdb_path)

layers.columns = ["layer_name", "geometry_type"]
display(layers)

/home/creighton.jo/.conda/envs/dam-removal/lib/python3.11/site-packages/pyogrio/core.py:169: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiLineString' is converted to 'MultiLineString Z'
  return ogr_list_layers(get_vsi_path_or_buffer(path_or_buffer))


,layer_name,geometry_type
0,ExternalCrosswalk,NaN
1,FeatureToMetadata,NaN
2,HUMod,NaN
3,NHDFCode,NaN
4,NHDFeatureToMetadata,NaN
...,...,...
81,N_1_JDesc,NaN
82,N_1_JStatus,NaN
83,N_1_JTopo,NaN
84,N_1_JTopo2,NaN


## List features for NHDFlowline layer

In [5]:
stream_layer = "NHDFlowline"

streams_sample = gpd.read_file(
    gdb_path,
    layer=stream_layer,
    rows=5,
)

print("CRS:", streams_sample.crs)
print("Geometry types:", streams_sample.geom_type.unique())
print("Sample dimensions:", streams_sample.shape)

print("\nColumns:")
print(streams_sample.columns.tolist())

display(streams_sample.drop(columns="geometry", errors="ignore").T)

CRS: EPSG:4269
Geometry types: <StringArray>
['MultiLineString']
Length: 1, dtype: str
Sample dimensions: (5, 19)

Columns:
['Permanent_Identifier', 'FDate', 'Resolution', 'GNIS_ID', 'GNIS_Name', 'LengthKM', 'ReachCode', 'FlowDir', 'WBArea_Permanent_Identifier', 'FType', 'FCode', 'MainPath', 'InNetwork', 'VisibilityFilter', 'Shape_Length', 'NHDPlusID', 'VPUID', 'Enabled', 'geometry']


/home/creighton.jo/.conda/envs/dam-removal/lib/python3.11/site-packages/pyogrio/raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiLineString' is converted to 'MultiLineString Z'
  return ogr_read(


,0,1,2,3,4
Permanent_Identifier,41703627,{4d36995d-3110-4258-b127-0dd6192a996d},41703639,{ff08e108-d790-4911-8f3d-aa8888415d67},{72b648ac-f097-4c6e-b537-d2d816ce734e}
FDate,2017-09-30 00:00:00+00:00,2017-09-30 00:00:00+00:00,2017-09-30 00:00:00+00:00,2017-09-30 00:00:00+00:00,2017-09-30 00:00:00+00:00
Resolution,2,2,2,2,2
GNIS_ID,None,None,None,None,None
GNIS_Name,None,None,None,None,None
LengthKM,3.09339,0.226673,0.576932,0.282999,0.071434
ReachCode,19020401000375,19020401005474,19020401000379,19020401003818,19020401006471
FlowDir,1,1,1,1,1
WBArea_Permanent_Identifier,None,None,None,None,None
FType,460,460,460,460,460


In [6]:
# Examine feature types, identifiers, and missing values
attributes = streams_sample.drop(columns="geometry", errors="ignore")

summary = pd.DataFrame({
    "dtype": attributes.dtypes.astype(str),
    "non_null_sample": attributes.notna().sum(),
    "unique_sample": attributes.nunique(dropna=True),
})

display(summary)

,dtype,non_null_sample,unique_sample
Permanent_Identifier,str,5,5
FDate,"datetime64[ms, UTC]",5,1
Resolution,int32,5,1
GNIS_ID,object,0,0
GNIS_Name,object,0,0
LengthKM,float64,5,5
ReachCode,str,5,5
FlowDir,int32,5,1
WBArea_Permanent_Identifier,object,0,0
FType,int32,5,1


## Show stream layer with respective watershed, dam removal(s), and water quality station(s):

In [18]:
ROOT = Path("/courses/DS5500.202710/teams/team4")
huc8 = "19020401"
station_file = ROOT / "data/raw/wqp/results/pilot_station_selection.csv"
dam_file = next(ROOT.rglob("ARDamRemovalList_Figshare_Mar2026.csv"))
wbd_path = next(ROOT.rglob("WBD_National_GPKG.gpkg"))

# Load HUC8 watershed boundary.
wbd_layers = gpd.list_layers(wbd_path)
hu8_layer = next(
    name for name in wbd_layers["name"]
    if "hu8" in name.lower()
)

watersheds = gpd.read_file(wbd_path, layer=hu8_layer)

huc_col = next(
    c for c in watersheds.columns if c.lower() == "huc8"
)

watershed = watersheds[
    watersheds[huc_col].astype(str).str.zfill(8) == huc8
].to_crs("EPSG:4326")

if watershed.empty:
    raise ValueError(f"HUC8 {huc8} not found")

print("Watershed loaded:", huc8)

Watershed loaded: 19020401


In [23]:
# Load all WQP station files.
WQP_DIR = ROOT / "data" / "raw" / "wqp"
STATION_DIR = WQP_DIR / "stations"

station_files = sorted(STATION_DIR.glob("*.csv"))
frames = []

for f in station_files:
    df = pd.read_csv(f, low_memory=False)
    df["source_huc8"] = f.stem
    frames.append(df)

if not frames:
    raise ValueError(f"No station CSVs found in {STATION_DIR}")

stations = pd.concat(frames, ignore_index=True)

# Keep surface-water stations.
surface_water_types = ["Stream", "River/Stream"]

surface_stations = stations[
    stations["MonitoringLocationTypeName"].isin(surface_water_types)
].copy()

# Validate coordinates and remove duplicate stations.
stations_map = surface_stations.copy()

stations_map["LatitudeMeasure"] = pd.to_numeric(
    stations_map["LatitudeMeasure"], errors="coerce"
)
stations_map["LongitudeMeasure"] = pd.to_numeric(
    stations_map["LongitudeMeasure"], errors="coerce"
)

stations_map = stations_map[
    stations_map["LatitudeMeasure"].between(-90, 90)
    & stations_map["LongitudeMeasure"].between(-180, 180)
].copy()

stations_map = stations_map.drop_duplicates(
    subset="MonitoringLocationIdentifier"
)

# Keep all stations geographically inside the pilot watershed.
stations_gdf = gpd.GeoDataFrame(
    stations_map,
    geometry=gpd.points_from_xy(
        stations_map["LongitudeMeasure"],
        stations_map["LatitudeMeasure"],
    ),
    crs="EPSG:4326",
)

stations_huc8 = gpd.sjoin(
    stations_gdf,
    watershed[["geometry"]],
    how="inner",
    predicate="within",
).drop(columns="index_right", errors="ignore")

print(f"Total surface-water stations: {len(stations_map):,}")
print(f"Stations inside HUC8 {huc8}: {len(stations_huc8):,}")

Total surface-water stations: 2,501
Stations inside HUC8 19020401: 391


In [24]:
dams = pd.read_csv(dam_file, low_memory=False)

dam_lat_col = "Latitude"
dam_lon_col = "Longitude"
dam_name_col = "Dam Name"

dams[dam_lat_col] = pd.to_numeric(
    dams[dam_lat_col], errors="coerce"
)
dams[dam_lon_col] = pd.to_numeric(
    dams[dam_lon_col], errors="coerce"
)

dams_geo = gpd.GeoDataFrame(
    dams.dropna(subset=[dam_lat_col, dam_lon_col]).copy(),
    geometry=gpd.points_from_xy(
        dams.dropna(subset=[dam_lat_col, dam_lon_col])[dam_lon_col],
        dams.dropna(subset=[dam_lat_col, dam_lon_col])[dam_lat_col],
    ),
    crs="EPSG:4326",
)

dams_huc8 = gpd.sjoin(
    dams_geo,
    watershed[["geometry"]],
    how="inner",
    predicate="within",
).drop(columns="index_right", errors="ignore")

print("Dam removals:", len(dams_huc8))

Dam removals: 1


In [27]:
import folium

# Load all streams from the already-extracted HUC8 geodatabase.
streams = gpd.read_file(gdb_path, layer=stream_layer)

# Clip streams to the target watershed.
streams = streams.to_crs(watershed.crs)
streams = gpd.clip(streams, watershed)

# Simplify stream geometry for faster map rendering.
streams_map = streams[["geometry"]].copy()
streams_map["geometry"] = streams_map.geometry.simplify(
    tolerance=0.00005,
    preserve_topology=True,
)

# Center map on watershed.
center = watershed.geometry.union_all().centroid

m = folium.Map(
    location=[center.y, center.x],
    zoom_start=10,
    tiles="OpenStreetMap",
)

# Watershed boundary.
# Convert datetime columns to strings for JSON serialization.
watershed_map = watershed.copy()

for col in watershed_map.columns:
    if pd.api.types.is_datetime64_any_dtype(watershed_map[col]):
        watershed_map[col] = watershed_map[col].astype(str)

folium.GeoJson(
    watershed_map.to_json(),
    name="HUC8 watershed",
    style_function=lambda feature: {
        "color": "blue",
        "weight": 2,
        "fillOpacity": 0.05,
    },
).add_to(m)

# All streams in the HUC8.
folium.GeoJson(
    streams_map.to_json(),
    name="NHDPlus streams",
    style_function=lambda feature: {
        "color": "cyan",
        "weight": 1,
    },
).add_to(m)

# All water-quality stations selected for this HUC8.
station_group = folium.FeatureGroup(name="Water-quality stations")

for _, row in stations_huc8.iterrows():
    folium.CircleMarker(
        location=[
            row["LatitudeMeasure"],
            row["LongitudeMeasure"],
        ],
        radius=4,
        color="green",
        fill=True,
        fill_opacity=0.8,
        tooltip=str(row["MonitoringLocationIdentifier"]),
    ).add_to(station_group)

station_group.add_to(m)

# All dam removals inside the watershed.
dam_group = folium.FeatureGroup(name="Dam removals")

for _, row in dams_huc8.iterrows():
    name = row.get(dam_name_col, "Unknown dam")

    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        tooltip=str(name),
        icon=folium.Icon(color="red", icon="tint"),
    ).add_to(dam_group)

dam_group.add_to(m)

folium.LayerControl().add_to(m)

# Fit map to watershed bounds.
minx, miny, maxx, maxy = watershed.total_bounds
m.fit_bounds([[miny, minx], [maxy, maxx]])

print("Streams:", len(streams))
print("Stations:", len(stations_huc8))
print("Dam removals:", len(dams_huc8))

m

/home/creighton.jo/.conda/envs/dam-removal/lib/python3.11/site-packages/pyogrio/raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiLineString' is converted to 'MultiLineString Z'
  return ogr_read(


Streams: 4954
Stations: 391
Dam removals: 1
